# is_rule judge — test bench

Calls the Perplexity **Agent API** directly (`requests`, no litellm).
System prompt = your rubric + few-shot examples parsed from `50 sample in prompt.json`.
User message = a batch of unlabelled rules from `50 control set.json`.


In [49]:
import json, os, re, time, requests

for ln in open(".env"):                       # PERPLEXITY_API_KEY
    if "=" in ln and not ln.startswith("#"):
        k, v = ln.strip().split("=", 1); os.environ.setdefault(k, v.strip('"\''))
KEY = os.environ["PERPLEXITY_API_KEY"]

MODEL       = "openai/gpt-5.6-terra"   # luna = cheaper, sol = stronger
EFFORT      = "minimal"                # minimal | low | medium | high | xhigh | max
                                       # ("none" is rejected by the API; minimal = 0 reasoning tokens)
SERVICE_TIER= "flex"                   # flex = 0.5x price, slower
BATCH_SIZE  = 5                        # rules per call
N_EXAMPLES  = 10                       # few-shot examples from the prompt set

from pathlib import Path
def D(name):                           # data files live in data/ since the reorg
    for q in (Path("data")/name, Path(name), Path("..")/"data"/name):
        if q.exists(): return q
    raise FileNotFoundError(name)

examples = json.load(open(D("50 sample in prompt.json")))[:N_EXAMPLES]
control  = json.load(open(D("50 control set.json")))
print(f"{len(examples)} examples, {len(control)} control rules, batch={BATCH_SIZE}")


10 examples, 50 control rules, batch=5


## 1 · System prompt — edit freely

The examples are appended automatically below your rubric.


In [50]:
RUBRIC = """
You classify clauses extracted from LLM agent instruction files (CLAUDE.md, AGENTS.md,
SKILL.md, .cursorrules, ...). Decide whether each clause is a RULE.
You will be given the Natural language clause, its heading path, and some aurrounding context. 
Both are useful information for deciding if the clause is a rule.

Defining is_rule vs non_rule:

A clause is a rule if it falls into the following taxonomy:

Prohibition: forbidding some action or pattern
Preference: directs a choice among options ("prefer X over Y")
Prescription: a standing convention as how something must be done, named, formatted, structured, or validated (Important distinction against one shot request or workflow action)
Permission: grants or withholds authority

A clause is NOT a rule if it is:
  Metadata: a name, tag, version, model id, or field value
  Description: what a thing is or does; a definition
  Explanation: why something is the case; rationale
  Context: when a tool, skill, or file applies; background
  Options: alternatives presented without directing a choice
  Procedure: one step inside an ordered, one-time task
  One-shot: a single request or diagnostic action, not a standing expectation

DECIDING
1. Would this still apply the next time the situation arises?
   No → not a rule.
2. Does it constrain behavior, or merely inform?
   Informs → not a rule.
3. Mood does not decide. A constraint stated as fact is still a rule ("IDs are case-sensitive", "the timeout is 30s", "names must be lowercase"). Ask what behavior the fact forces. If it forces exactly one, it is a rule; if it merely informs, it is not.

Return ONLY a JSON array, one object per input rule, no prose:
[{"ID": <int>, "is_rule": <true|false>}]
"""

def render(ex):
    return f'ID: {ex["ID"]}\nRULE: {ex["rule"]}\nCONTEXT:\n{ex["context"]}'

SYSTEM = (RUBRIC.strip() + "\n\n## Examples\n\n" +
          "\n\n".join(render(e) + f'\n→ is_rule: {str(e["is_rule"]).lower()}'
                        + (f'  ({e["reason"]})' if e["reason"] else "")
                        for e in examples))
print(f"system prompt: {len(SYSTEM):,} chars  (~{len(SYSTEM)//4:,} tok)")
print(SYSTEM[:600] + "\n...")


system prompt: 7,280 chars  (~1,820 tok)
You classify clauses extracted from LLM agent instruction files (CLAUDE.md, AGENTS.md,
SKILL.md, .cursorrules, ...). Decide whether each clause is a RULE.
You will be given the Natural language clause, its heading path, and some aurrounding context. 
Both are useful information for deciding if the clause is a rule.

Defining is_rule vs non_rule:

A clause is a rule if it falls into the following taxonomy:

Prohibition: forbidding some action or pattern
Preference: directs a choice among options ("prefer X over Y")
Prescription: a standing convention as how something must be done, named, format
...


## 2 · The call


## 2 · The call

The whole 50-row control set goes in; `BATCH_SIZE` decides how it is chunked,
and chunks are sent in parallel under a **30 requests/second** ceiling.


In [51]:
API = "https://api.perplexity.ai/v1/agent"
RPS = 30                                    # hard ceiling on request rate

def build(batch):
    user = "\n\n".join(render(r) for r in batch)
    return user, {"model": MODEL, "instructions": SYSTEM, "input": user,
                  "reasoning": {"effort": EFFORT}, "max_output_tokens": 2000,
                  "service_tier": SERVICE_TIER, "prompt_cache_key": "is-rule-v1"}

def judge(batch, show=True):
    user, body = build(batch)
    if show:
        print("═" * 78); print("THE CALL"); print("═" * 78)
        print(f"model={MODEL}  effort={EFFORT}  tier={SERVICE_TIER}  batch={len(batch)}")
        print(f"system: {len(SYSTEM):,} chars   user: {len(user):,} chars\n")
        print("--- user message ---"); print(user[:1500] + ("..." if len(user) > 1500 else ""))
    t0 = time.time()
    r = requests.post(API, headers={"Authorization": f"Bearer {KEY}"}, json=body, timeout=180)
    r.raise_for_status()
    return r.json(), time.time() - t0

def text_of(resp):
    """Pull the assistant text out of the Responses-shaped payload."""
    if resp.get("output_text"): return resp["output_text"]
    return "\n".join(c["text"] for it in resp.get("output", [])
                      for c in (it.get("content") or []) if c.get("text"))

def parse(raw):
    m = re.search(r"\[.*\]", raw, re.S)
    return {int(d["ID"]): bool(d["is_rule"]) for d in json.loads(m.group(0))} if m else {}


## 3 · One batch — call, raw output, parsed


In [ ]:
batch = control[:BATCH_SIZE]
resp, secs = judge(batch)

print("\n" + "═" * 78); print("RAW OUTPUT"); print("═" * 78)
raw = text_of(resp); print(raw)
u = resp.get("usage") or {}
cached = (u.get("input_tokens_details") or {}).get("cached_tokens", 0)
print(f"\nin={u.get('input_tokens')}  cached={cached}  out={u.get('output_tokens')}  "
      f"cost=${(u.get('cost') or {}).get('total_cost', 0):.5f}  {secs:.1f}s  tier={resp.get('service_tier')}")

print("\n" + "═" * 78); print("PARSED"); print("═" * 78)
verdict = parse(raw)
for r in batch:
    v = verdict.get(r["ID"])
    print(f"  ID {r['ID']:>3}  {'RULE ' if v else 'not  ' if v is not None else '  ?  '}  {r['rule'][:78]}")


## 4 · Whole control set, streamed in parallel (≤30 rps)


In [52]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

_lock, _slots = threading.Lock(), []

def _throttle():
    """Block until firing now keeps the trailing 1s window under RPS."""
    while True:
        with _lock:
            now = time.time()
            _slots[:] = [t for t in _slots if now - t < 1.0]
            if len(_slots) < RPS:
                _slots.append(now); return
            wait = 1.0 - (now - _slots[0])
        time.sleep(max(wait, 0.01))

def run_all(rows, batch_size=None, workers=8):
    bs = batch_size or BATCH_SIZE
    chunks = [rows[i:i+bs] for i in range(0, len(rows), bs)]
    out, usage, t0 = {}, [], time.time()
    def one(ch):
        _throttle()
        resp, _ = judge(ch, show=False)
        return parse(text_of(resp)), (resp.get("usage") or {})
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(one, c): c for c in chunks}
        for n, f in enumerate(as_completed(futs), 1):
            v, u = f.result(); out.update(v); usage.append(u)
            print(f"  {n}/{len(chunks)} chunks · {len(out)}/{len(rows)} rules", end="\r")
    dt = time.time() - t0
    cost = sum((u.get("cost") or {}).get("total_cost", 0) for u in usage)
    cin  = sum((u.get("input_tokens_details") or {}).get("cached_tokens", 0) for u in usage)
    tin  = sum(u.get("input_tokens", 0) for u in usage)
    print(f"\n{len(chunks)} calls in {dt:.1f}s ({len(chunks)/dt:.1f} rps) · "
          f"cache {cin/tin:.0%} · ${cost:.4f} · ${cost/len(rows)*1e6/1e3:.2f} per 1k rules")
    return out

verdicts = run_all(control)


  10/10 chunks · 50/50 rules
10 calls in 8.1s (1.2 rps) · cache 54% · $0.0240 · $0.48 per 1k rules


## 5 · Score against the held-out answers


In [53]:
truth  = {r["ID"]: r["is_rule"] for r in json.load(open(D("50 control ANSWERS.json")))}
by_id  = {r["ID"]: r for r in control}
hit    = [i for i in truth if verdicts.get(i) == truth[i]]
tp = sum(1 for i in truth if verdicts.get(i) and truth[i])
fp = sum(1 for i in truth if verdicts.get(i) and not truth[i])
fn = sum(1 for i in truth if verdicts.get(i) is False and truth[i])
print(f"accuracy  {len(hit)}/{len(truth)} = {len(hit)/len(truth):.0%}")
print(f"precision {tp/(tp+fp):.0%}   recall {tp/(tp+fn):.0%}   (tp={tp} fp={fp} fn={fn})")
print(f"\nmisses ({len(truth)-len(hit)}):")
for i in sorted(truth):
    if verdicts.get(i) != truth[i]:
        print(f"  ID {i:>3}  judge={str(verdicts.get(i)):<5} truth={str(truth[i]):<5} {by_id[i]['rule'][:66]}")


accuracy  48/50 = 96%
precision 92%   recall 100%   (tp=22 fp=2 fn=0)

misses (2):
  ID  43  judge=True  truth=False A scene-windowing workaround is a consumer band-aid, not an engine
  ID  70  judge=True  truth=False - `en.json` - English (required base language)
